In [1]:
#Pre-posted codes

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/titanic/train.csv
/kaggle/input/titanic/test.csv
/kaggle/input/titanic/gender_submission.csv


In [2]:
#save training data from filepath to variable

train_data = pd.read_csv("/kaggle/input/titanic/train.csv")
train_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
#save testing data from filepath to variable

test_data = pd.read_csv("/kaggle/input/titanic/test.csv")
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [4]:
#list the existing columns in training data (not required, just following trail of thought)

list(train_data.columns)

['PassengerId',
 'Survived',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

In [5]:
#drop name column

edited_train_data = train_data.drop(labels="Name",axis=1)
edited_test_data = test_data.drop(labels="Name",axis=1)

In [6]:
#identify columns with missing values and the number of its respective missing values
#notice the columns instead of columns() to avoid the callable error

print("total number of rows and columns " + str(edited_train_data.shape))

missing_count = edited_train_data.isnull().sum()
print(missing_count[missing_count > 0])

total number of rows and columns (891, 11)
Age         177
Cabin       687
Embarked      2
dtype: int64


In [7]:
#just some basic stats for my own satisfaction

missing_age = 177
missing_cabin = 687
missing_embarked = 2

numrow = 891

print("{} or {:.1f}% of age data missing".format(missing_age,missing_age/numrow*100))
print("{} or {:.1f}% of cabin data missing".format(missing_cabin,missing_cabin/numrow*100))
print("{} or {:.1f}% of embarked data missing".format(missing_embarked,missing_embarked/numrow*100))

#drop the cabin data

edited_train_data = edited_train_data.drop(labels = "Cabin", axis = 1)
edited_test_data = edited_test_data.drop(labels = "Cabin", axis = 1)

177 or 19.9% of age data missing
687 or 77.1% of cabin data missing
2 or 0.2% of embarked data missing


In [8]:
#count number of unique values per column

edited_train_data.nunique()

PassengerId    891
Survived         2
Pclass           3
Sex              2
Age             88
SibSp            7
Parch            7
Ticket         681
Fare           248
Embarked         3
dtype: int64

In [9]:
#drop ticket as most of values are unique and "seems" difficult to classify into categories useful for a model

edited_train_data = edited_train_data.drop(labels = "Ticket", axis = 1)
edited_test_data = edited_test_data.drop(labels = "Ticket", axis = 1)

In [10]:
#list the columns containing variables dtype=object

object_cols = [col for col in edited_train_data.columns if edited_train_data[col].dtype=="object"]
print(object_cols)

['Sex', 'Embarked']


In [11]:
#list the different variables from the columns above

print(edited_train_data["Sex"].unique())
print(edited_train_data["Embarked"].unique())

['male' 'female']
['S' 'C' 'Q' nan]


In [12]:
#let's look at and fix the 2 missing variables from Embarked first
#maybe not necessarily the most accurate way to go about it, but 0.2% error is an acceptable range, so going for the simplest solution by replacing it with the mode
#note the [0] at the end of the first line of code to make this work

edited_train_data["Embarked"] = edited_train_data["Embarked"].fillna(edited_train_data["Embarked"].mode()[0])
edited_test_data["Embarked"] = edited_test_data["Embarked"].fillna(edited_test_data["Embarked"].mode()[0])

In [13]:
#using the OrdinalEncoder on just the columns containing object data

from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
edited_train_data[object_cols] = ordinal_encoder.fit_transform(edited_train_data[object_cols])
edited_test_data[object_cols] = ordinal_encoder.fit_transform(edited_test_data[object_cols])

In [14]:
#we still have the 177 missing values from age that needs to be cleaned up
#let's (intuitively) try the mean from SimpleImputer

from sklearn.impute import SimpleImputer

my_imputer = SimpleImputer()

imputed_train_data = pd.DataFrame(my_imputer.fit_transform(edited_train_data))
imputed_test_data = pd.DataFrame(my_imputer.fit_transform(edited_test_data))

imputed_train_data.columns = edited_train_data.columns
imputed_test_data.columns = edited_test_data.columns

In [15]:
#count number of unique values per column (once again, but this time for cardinality assessment)

edited_train_data.nunique()

PassengerId    891
Survived         2
Pclass           3
Sex              2
Age             88
SibSp            7
Parch            7
Fare           248
Embarked         3
dtype: int64

In [16]:
#so, up until now, it's been about cleaning up data - and some of it is PROBABLY VERY unneccessary - but for the sake of learning...

from sklearn.ensemble import RandomForestClassifier

y = imputed_train_data["Survived"]

features = ["Embarked", "Sex", "Pclass", "Parch"]
X = pd.get_dummies(imputed_train_data[features])
X_test = pd.get_dummies(imputed_test_data[features])

model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)
model.fit(X, y)
predictions = model.predict(X_test)

output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
